# Assignment 3: SLCM Classification Analysis
**CS-430-01 Machine Learning | Programming Assignments**

| | |
|---|---|
| **Format** | Individual take-home Jupyter/Colab notebook |
| **Dataset** | SLCM_rent_arrears_student_clean.csv (same dataset as Assignment 2) |
| **Target variable** | `high_arrears`: 1 if `past_due_rent` is $3,000 or more; otherwise 0 |
| **Models** | Logistic Regression, KNN, Decision Tree, Random Forest, SVC |
| **Submission** | GitHub repository link submitted in Moodle |
| **Class follow-up** | Whole-class discussion; everyone participates |
| **Points** | 100 |

## Goal

In Assignment 2, you predicted the amount of overdue rent. Now the question is: Can household and housing information help identify households with $3,000 or more in past-due rent? Apply five classification models using the same prepared data. Focus on a fair comparison and a clear explanation of the results rather than complicated tuning.

## Notebook Instructions

- Copy each numbered question into a Markdown cell; answer beneath it in small code or Markdown cells.

- After important outputs and the chart, write 1–3 sentences of interpretation.

- Use the same 80/20 train/test split for all five models: `test_size=0.20, random_state=42, stratify=y`.

- Use the same preprocessing pipeline for each model; you may reuse the example from class.

- Do not use `record_id` or `past_due_rent` as model inputs. Run all cells and keep outputs visible.

## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# connect to drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Part 1 – Understand the Classification Problem (15 points)

## 1. Explore the dataset (7 points)

- Load the CSV. Display five rows, the data shape, the column names, and missing-value counts.

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Assignment Three/SLCM_rent_arrears_student_clean.csv')
df.head()

,record_id,adults_in_household,children_in_household,household_has_65plus,monthly_household_income,income_sources,employment_status,has_housing_voucher,eviction_in_progress,behind_on_utilities,main_reason_behind_on_rent,intake_quarter,monthly_rent,past_due_rent
0,SLCM_0001,1,1,No,293.0,"Food benefits - TANF/SNAP, No income, Other",Unemployed but want to find work,No,I don't know,Yes,Other,2026-Q1,1560.0,3100.0
1,SLCM_0002,0,3,No,2000.0,Child support,Employed,No,I don't know,Yes,Unexpected expense,2026-Q3,1550.0,7300.0
2,SLCM_0003,1,0,No,400.0,Food benefits - TANF/SNAP,Unemployed but want to find work,I don't know,Yes,Yes,Lost a job/had hours cut,2025-Q4,0.0,6000.0
3,SLCM_0004,2,0,No,2500.0,Full-time employment,Unemployed and not looking for work,No,Yes,Yes,Lost a job/had hours cut,2026-Q2,1299.0,2500.0
4,SLCM_0005,1,2,No,14.0,Part-time employment,Employed,No,Yes,No,Unexpected expense,2025-Q4,1105.5,1205.5


In [4]:
df.shape

(1754, 14)

In [5]:
df.columns

Index(['record_id', 'adults_in_household', 'children_in_household',
       'household_has_65plus', 'monthly_household_income', 'income_sources',
       'employment_status', 'has_housing_voucher', 'eviction_in_progress',
       'behind_on_utilities', 'main_reason_behind_on_rent', 'intake_quarter',
       'monthly_rent', 'past_due_rent'],
      dtype='object')

In [6]:
df.isnull().sum()

,0
record_id,0
adults_in_household,0
children_in_household,0
household_has_65plus,0
monthly_household_income,108
income_sources,0
employment_status,0
has_housing_voucher,0
eviction_in_progress,1
behind_on_utilities,7


- Identify one numerical and one categorical feature.

Numerical Feature: `monthly_rent`  
Categorical Feature: `employement_status`

- Answer: Why must `past_due_rent` be excluded from the input features?

`past_due_rent` must be excluded because it would allow the model to "cheat" and see the information its trying to predict when it's trying to predict it. If it can see that column it can directly see that the `past_due_rent` is either above or below $3k.  
This is also called data/information leakage and can also occur by things such as mixing training and test samples during training.

## 2. Create the target (8 points)

- Create `high_arrears`: 1 if `past_due_rent` is at least $3,000, otherwise 0.

In [7]:
df['high_arrears'] = (df['past_due_rent'] >= 3000).astype(int)

- Show class counts and one bar chart of `high_arrears`.

In [8]:
df['high_arrears'].value_counts()

,count
high_arrears,
0,1210
1,544


- Answer: Which class is larger? Why might accuracy alone be misleading?

The class of renters with rent less than $3k monthly is larger.   
Accuracy could be misleading here because predicting all class 0 would give a higher than random accuracy (544/1210 vs 1/2) even if it fails to detect the smaller class in all or many cases.

# Part 2 – Prepare the Data (25 points)

## 3. Select features and split (10 points)

- Create `X` by dropping `record_id`, `past_due_rent`, and `high_arrears`. Set `y = high_arrears`.

In [9]:
X = df.drop(columns=['record_id', 'past_due_rent', 'high_arrears'])
y = df['high_arrears']

- Split the data once into training and testing sets.

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

- Answer: Why do we keep a separate test set?

We keep the test set separate so information from those samples doesn't leak into our training data during data imputation. Eg: computing the median to impute for missing values should only use the training values to compute the median to prevent data leakage.

## 4. Build one reusable preprocessor (15 points)

- Identify numerical and categorical columns with `select_dtypes`.

In [11]:
numerical_columns = X.select_dtypes(include='number').columns
categorical_columns = X.select_dtypes(exclude='number').columns

- Numerical: fill missing values using median and apply `StandardScaler`.

In [12]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numerical_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

- Categorical: fill missing values using most_frequent and apply `OneHotEncoder(handle_unknown="ignore")`.

In [13]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

- Combine with `ColumnTransformer`. You may reuse the class example.

In [14]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_columns),
    ('categorical', categorical_pipeline, categorical_columns)
])

- Answer: Why is scaling helpful for KNN and SVC?

Scaling is helpful for KNN because the nearest neighbors are calculated by finding the k nearest points based on their euclidian distances, which allows for large features to dominate the distance calculation. Feature scaling prevents this.  
SVC also uses euclidian-like distance calculations so large features can dominate calculations there too.

# Part 3 – Train and Compare Five Models (45 points)

## 5. Train the five classifiers (20 points)

- Build five pipelines with the same preprocessor, the same training set, and these models:

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

- Logistic Regression: `LogisticRegression(max_iter=1000)`

In [16]:
logistic_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

- KNN: `KNeighborsClassifier(n_neighbors=5)`

In [17]:
knn_pipeline = make_pipeline(
    preprocessor,
    KNeighborsClassifier(n_neighbors=5)
)

- Decision Tree: `DecisionTreeClassifier(max_depth=5, random_state=42)`

In [18]:
decision_tree_pipeline = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

- Random Forest: `RandomForestClassifier(n_estimators=100, random_state=42)`

In [19]:
random_forest_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(n_estimators=100, random_state=42)
)

- Support Vector Classifier: `SVC()`

In [20]:
svc_model = make_pipeline(
    preprocessor,
    SVC()
)

- Fit each pipeline and report test accuracy, recall, and F1 for class 1. Use a short, repeated code pattern; you do not need a loop or custom functions.

In [21]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

logistic_model = logistic_pipeline.fit(X_train, y_train)
logistic_y_pred = logistic_model.predict(X_test)
print(f"Logstic Accuracy: {accuracy_score(y_test, logistic_y_pred)}")
print(f"Logistic Recall: {recall_score(y_test, logistic_y_pred)}")
print(f"Logistic F1: {f1_score(y_test, logistic_y_pred)}")
print()
knn_model = knn_pipeline.fit(X_train, y_train)
knn_y_pred = knn_model.predict(X_test)
print(f"KNN Accuracy: {accuracy_score(y_test, knn_y_pred)}")
print(f"KNN Recall: {recall_score(y_test, knn_y_pred)}")
print(f"KNN F1: {f1_score(y_test, knn_y_pred)}")
print()
decision_tree_model = decision_tree_pipeline.fit(X_train, y_train)
decision_tree_y_pred = decision_tree_model.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, decision_tree_y_pred)}")
print(f"Decision Tree Recall: {recall_score(y_test, decision_tree_y_pred)}")
print(f"Decision Tree F1: {f1_score(y_test, decision_tree_y_pred)}")
print()
random_forest_model = random_forest_pipeline.fit(X_train, y_train)
random_forest_y_pred = random_forest_model.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, random_forest_y_pred)}")
print(f"Random Forest Recall: {recall_score(y_test, random_forest_y_pred)}")
print(f"Random Forest F1: {f1_score(y_test, random_forest_y_pred)}")
print()
svc_model = svc_model.fit(X_train, y_train)
svc_y_pred = svc_model.predict(X_test)
print(f"SVC Accuracy: {accuracy_score(y_test, svc_y_pred)}")
print(f"SVC Recall: {recall_score(y_test, svc_y_pred)}")
print(f"SVC F1: {f1_score(y_test, svc_y_pred)}")

Logstic Accuracy: 0.6951566951566952
Logistic Recall: 0.30275229357798167
Logistic F1: 0.3815028901734104

KNN Accuracy: 0.6809116809116809
KNN Recall: 0.3761467889908257
KNN F1: 0.422680412371134

Decision Tree Accuracy: 0.6951566951566952
Decision Tree Recall: 0.3302752293577982
Decision Tree F1: 0.4022346368715084

Random Forest Accuracy: 0.7037037037037037
Random Forest Recall: 0.3211009174311927
Random Forest F1: 0.40229885057471265

SVC Accuracy: 0.7065527065527065
SVC Recall: 0.22935779816513763
SVC F1: 0.32679738562091504


## 6. Compare the models (10 points)

- Put the test results in one table with columns: Model, Accuracy, Recall, F1.

In [22]:
models_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN', 'Decision Tree', 'Random Forest', 'SVC'],
    'Accuracy': [accuracy_score(y_test, logistic_y_pred), accuracy_score(y_test, knn_y_pred), accuracy_score(y_test, decision_tree_y_pred), accuracy_score(y_test, random_forest_y_pred), accuracy_score(y_test, svc_y_pred)],
    'Recall': [recall_score(y_test, logistic_y_pred), recall_score(y_test, knn_y_pred), recall_score(y_test, decision_tree_y_pred), recall_score(y_test, random_forest_y_pred), recall_score(y_test, svc_y_pred)],
    'F1': [f1_score(y_test, logistic_y_pred), f1_score(y_test, knn_y_pred), f1_score(y_test, decision_tree_y_pred), f1_score(y_test, random_forest_y_pred), f1_score(y_test, svc_y_pred)]
})
models_df

,Model,Accuracy,Recall,F1
0,Logistic Regression,0.695157,0.302752,0.381503
1,KNN,0.680912,0.376147,0.422680
2,Decision Tree,0.695157,0.330275,0.402235
3,Random Forest,0.703704,0.321101,0.402299
4,SVC,0.706553,0.229358,0.326797


- Create one confusion matrix for the model with the highest test F1.

In [23]:
from sklearn.metrics import confusion_matrix

best_f1_model = knn_model
y_pred = best_f1_model.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[198,  44],
       [ 68,  41]])

- Answer: Which model catches the highest proportion of high-arrears households? What does a missed case mean for SLCM?

The model that catches the most high-arrears is the KNN model because it has the highest recall score, which calculates `true_positive`/(`true_positive`+`false_negative`) which explicitly punishes missed positives and ignores false positives.  

For SLCM, a missed case means a client actually was more than \$3k of rent past due, but our model predicted that they were less than \$3k of rent past due.

## 7. Try a small GridSearchCV (15 points)

- Tune only two models on the training set with `cv=3` and `scoring="f1"`.

In [24]:
from sklearn.model_selection import GridSearchCV

- For KNN, try `n_neighbors = [3, 5, 7]`.

In [25]:
knn_param_grid = {
    'kneighborsclassifier__n_neighbors': [3, 5, 7]
}
knn_grid_search = GridSearchCV(knn_pipeline, knn_param_grid, cv=3, scoring='f1')

- For Random Forest, try `max_depth = [3, 5, None]` (keep `n_estimators=100`).

In [26]:
random_forest_param_grid = {
    'randomforestclassifier__max_depth': [3, 5, None]
}
random_forest_grid_search = GridSearchCV(random_forest_pipeline, random_forest_param_grid, cv=3, scoring='f1')

- Print `best_params_` and `best_score_` for each search. No extra charts or second search are needed.

In [27]:
knn_grid_search.fit(X_train, y_train)
best_knn_model = knn_grid_search.best_estimator_
print(f"Best KNN Params: {knn_grid_search.best_params_}")
print(f"Best KNN Score: {knn_grid_search.best_score_}")
print()
random_forest_grid_search.fit(X_train, y_train)
best_random_forest_model = random_forest_grid_search.best_estimator_
print(f"Best Random Forest Params: {random_forest_grid_search.best_params_}")
print(f"Best Random Forest Score: {random_forest_grid_search.best_score_}")

Best KNN Params: {'kneighborsclassifier__n_neighbors': 7}
Best KNN Score: 0.4854649923374277

Best Random Forest Params: {'randomforestclassifier__max_depth': None}
Best Random Forest Score: 0.4574743145651916


- Answer: Which has the higher best cross-validation F1? Why is GridSearchCV fitted on training data only?

The KNN model has the better f1 score (0.485 vs 0.457), and GridSearchCV only uses training data because we still want to be able to maintain isolation of training and test data to prevent leakage.   
Within the GridSearch each fold separates its data into training and validation data, that way each model can be scored without losing the integrity of the testing data set.

# Part 4 – Explain the Results (15 points)

## 8. Interpret your comparison (8 points)

- In 4–5 sentences, identify the most promising model based on your results. Explain your choice using F1 and recall, not accuracy alone.

- Did a more complex model necessarily perform better? What types of errors should SLCM pay attention to?

The model comparison dataframe below indicates that the most promising model is the original knn model, where n_neighbors=5.  

Even though this model had a slightly lower accuracy score than some others (~68% vs top accuracy of ~70.6%), it was able to outpreform all of the other models in both f1-score and recall score. The knn model has an f1-score ~4% higher than the 2nd best and a ~2% higher recall score.

In this case, the more complex model is not necessarily the most complex one, as our tuned knn model used parameters of n_neighbors=7 and managed to achieve a slightly higher accuracy score, but at the cost of f1 and recall score.
For SLCM specifically, accurately capturing the highest proportion of high-arrears appears to be a top priority, so recall would be the most important metric as its scoring punishes false negatives. These (false negatives) are the errors SLCM must pay close attention to.

In [28]:
tuned_knn_pred = best_knn_model.predict(X_test)
tuned_rf_pred = best_random_forest_model.predict(X_test)

tuned_models_results = pd.DataFrame({
    'Model': ['Tuned KNN', 'Tuned Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, tuned_knn_pred),
        accuracy_score(y_test, tuned_rf_pred)
    ],
    'Recall': [
        recall_score(y_test, tuned_knn_pred),
        recall_score(y_test, tuned_rf_pred)
    ],
    'F1': [
        f1_score(y_test, tuned_knn_pred),
        f1_score(y_test, tuned_rf_pred)
    ]
})

models_df = pd.concat([models_df, tuned_models_results], ignore_index=True)
models_df

,Model,Accuracy,Recall,F1
0,Logistic Regression,0.695157,0.302752,0.381503
1,KNN,0.680912,0.376147,0.422680
2,Decision Tree,0.695157,0.330275,0.402235
3,Random Forest,0.703704,0.321101,0.402299
4,SVC,0.706553,0.229358,0.326797
5,Tuned KNN,0.680912,0.302752,0.370787
6,Tuned Random Forest,0.703704,0.321101,0.402299


## 9. Recommendation to SLCM (7 points)

- Write 4–5 sentences for a non-technical SLCM audience: what you tried to predict, how well the selected model performed, one practical insight, and one limitation.

- State that a prediction is not proof of cause and must not be used alone to decide eligibility for assistance.

In this analysis, we built machine learning models to identify households at risk of falling into high rent arrears (specifically past-due rent of $3,000 or more).

The selected model (KNN) was able to successfully identify whether a household was a high arrear household or not about 70% of the time, which is better than random guessing.  

However, a major limitation is that even with this high accuracy score, the model misses roughly 62% of the high-arrears cases (false negatives), meaning it is not a perfect safety net.
Most importantly, a model's prediction is just that, only a prediction. It should never be used a the singular reason accept or deny a households eligibility for assistance.


## Grading
| Part | Points |
|---|---|
| Part 1 – Understand the Problem | 15 |
| Part 2 – Prepare the Data | 25 |
| Part 3 – Train and Compare Models | 45 |
| Part 4 – Explain Results | 15 |
| **Total** | **100** |

## What to Submit

- Completed .ipynb notebook with questions, code, interpretations, and visible outputs.

- Provided CSV and a short README in your GitHub repository. Submit the GitHub link in Moodle.

## Responsible Use of Data and AI

Respect the de-identified community data. Do not attempt to identify households. AI may help explain or debug code, but you must understand, verify, and disclose AI use. Your interpretations and class discussion must reflect your own understanding.